# Fire Simple Extension 3 — Narrativa e Simulação em Python

Este notebook reproduz, em Python, a lógica do modelo **Fire Simple Extension 3**, inspirado no modelo de propagação de incêndio em floresta do NetLogo.

A simulação representa uma floresta em uma grade bidimensional, onde:

- cada célula pode estar vazia, conter uma árvore verde, estar em chamas ou já ter queimado;
- o fogo começa na borda esquerda do ambiente;
- a propagação ocorre para os quatro vizinhos diretos;
- a probabilidade de propagação é afetada pela direção e intensidade do vento;
- quando `big_jumps` está ativado, faíscas podem saltar para posições mais distantes, simulando transporte pelo vento.

O objetivo é observar como **densidade da floresta**, **probabilidade de propagação**, **vento** e **saltos de faíscas** alteram o comportamento global do incêndio.

## 1. Modelo original em NetLogo

O modelo original define uma variável global chamada `initial-trees`, responsável por registrar quantas árvores verdes existiam no início.

No procedimento `setup`, o ambiente é limpo, árvores são distribuídas aleatoriamente conforme a variável `density`, e a coluna da esquerda é incendiada.

No procedimento `go`, cada árvore em chamas verifica seus quatro vizinhos. Se houver árvores verdes, a chance de ignição é ajustada pelo vento:

- vento sul pode ajudar ou dificultar a propagação vertical;
- vento oeste pode ajudar ou dificultar a propagação horizontal;
- árvores queimadas ficam escurecidas;
- a simulação termina quando não há mais células vermelhas.

## 2. Estados da simulação

Nesta versão em Python, usaremos os seguintes valores:

| Valor | Estado |
|---:|---|
| 0 | célula vazia |
| 1 | árvore verde |
| 2 | árvore em chamas |
| 3 | árvore queimada |

A lógica é equivalente ao modelo NetLogo, mas adaptada para execução em notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 3. Parâmetros do modelo

Os principais parâmetros são:

- `density`: porcentagem de células ocupadas por árvores;
- `probability_of_spread`: probabilidade base de propagação do fogo;
- `south_wind_speed`: intensidade do vento sul;
- `west_wind_speed`: intensidade do vento oeste;
- `big_jumps`: ativa ou desativa saltos de faíscas;
- `width` e `height`: dimensões da grade.

In [ ]:
width = 80
height = 60

density = 58
probability_of_spread = 55

south_wind_speed = 10
west_wind_speed = 15

big_jumps = True
max_ticks = 300

## 4. Função de inicialização

A função `setup()` cria a floresta.

Primeiro, cada célula recebe uma árvore com probabilidade definida por `density`. Depois, a primeira coluna da esquerda é marcada como fogo inicial.

In [ ]:
EMPTY = 0
TREE = 1
FIRE = 2
BURNED = 3

def setup(width, height, density, rng):
    '''
    Inicializa a grade da floresta.

    Args:
        width: largura da grade
        height: altura da grade
        density: percentual de ocupação por árvores
        rng: gerador aleatório do NumPy

    Returns:
        grid: matriz com estados da simulação
        initial_trees: quantidade inicial de árvores verdes
    '''
    grid = np.zeros((height, width), dtype=np.int8)

    tree_mask = rng.random((height, width)) < (density / 100)
    grid[tree_mask] = TREE

    # Coluna esquerda inicia em chamas
    grid[:, 0] = FIRE

    initial_trees = np.sum(grid == TREE)
    return grid, initial_trees

grid, initial_trees = setup(width, height, density, rng)
initial_trees

## 5. Visualização da floresta

As cores seguem a interpretação do modelo:

- branco: vazio;
- verde: árvore;
- vermelho: fogo;
- preto: queimado.

In [ ]:
cmap = ListedColormap(["white", "green", "red", "black"])

def plot_grid(grid, title="Estado da simulação"):
    plt.figure(figsize=(10, 7))
    plt.imshow(grid, cmap=cmap, vmin=0, vmax=3)
    plt.title(title)
    plt.axis("off")
    plt.show()

plot_grid(grid, "Estado inicial da floresta")

## 6. Regras de propagação

Cada célula em chamas avalia seus quatro vizinhos diretos: norte, sul, leste e oeste.

A probabilidade de propagação começa em `probability_of_spread` e é ajustada conforme a posição do fogo em relação à árvore vizinha.

A implementação abaixo segue a mesma interpretação do código NetLogo:

- se o fogo está ao norte da árvore, o vento sul dificulta;
- se o fogo está ao sul da árvore, o vento sul facilita;
- se o fogo está a leste da árvore, o vento oeste dificulta;
- se o fogo está a oeste da árvore, o vento oeste facilita.

In [ ]:
def step_fire(
    grid,
    probability_of_spread,
    south_wind_speed,
    west_wind_speed,
    big_jumps,
    rng
):
    '''
    Executa um tick da simulação.

    Returns:
        new_grid: grade atualizada
        ignitions: número de novas ignições
    '''
    height, width = grid.shape
    new_grid = grid.copy()
    ignitions = 0

    burning_cells = np.argwhere(grid == FIRE)

    for row, col in burning_cells:
        # (delta_linha, delta_coluna, direção do fogo em relação ao vizinho)
        neighbors = [
            (-1,  0, 180),  # árvore ao norte; fogo está ao sul dela
            ( 1,  0,   0),  # árvore ao sul; fogo está ao norte dela
            ( 0, -1,  90),  # árvore a oeste; fogo está a leste dela
            ( 0,  1, 270),  # árvore a leste; fogo está a oeste dela
        ]

        for dr, dc, direction in neighbors:
            nr, nc = row + dr, col + dc

            if 0 <= nr < height and 0 <= nc < width and grid[nr, nc] == TREE:
                probability = probability_of_spread

                if direction == 0:
                    probability -= south_wind_speed
                elif direction == 90:
                    probability -= west_wind_speed
                elif direction == 180:
                    probability += south_wind_speed
                elif direction == 270:
                    probability += west_wind_speed

                probability = max(0, min(100, probability))

                if rng.integers(0, 100) < probability:
                    new_grid[nr, nc] = FIRE
                    ignitions += 1

                    if big_jumps:
                        jump_col = nc + int(round(west_wind_speed / 5))
                        jump_row = nr + int(round(south_wind_speed / 5))

                        if (
                            0 <= jump_row < height
                            and 0 <= jump_col < width
                            and grid[jump_row, jump_col] == TREE
                        ):
                            new_grid[jump_row, jump_col] = FIRE
                            ignitions += 1

        new_grid[row, col] = BURNED

    return new_grid, ignitions

## 7. Execução da simulação

A simulação avança até que não haja mais células em chamas ou até atingir `max_ticks`.

In [ ]:
def run_simulation(
    width=80,
    height=60,
    density=58,
    probability_of_spread=55,
    south_wind_speed=10,
    west_wind_speed=15,
    big_jumps=True,
    max_ticks=300,
    seed=42
):
    rng = np.random.default_rng(seed)
    grid, initial_trees = setup(width, height, density, rng)

    history = []
    snapshots = {}
    tick = 0

    while np.any(grid == FIRE) and tick < max_ticks:
        trees = np.sum(grid == TREE)
        burning = np.sum(grid == FIRE)
        burned = np.sum(grid == BURNED)

        history.append({
            "tick": tick,
            "trees": trees,
            "burning": burning,
            "burned": burned,
            "percent_burned": 100 * burned / max(initial_trees, 1)
        })

        if tick in [0, 5, 10, 20, 40, 80, 120]:
            snapshots[tick] = grid.copy()

        grid, ignitions = step_fire(
            grid,
            probability_of_spread,
            south_wind_speed,
            west_wind_speed,
            big_jumps,
            rng
        )

        tick += 1

    history.append({
        "tick": tick,
        "trees": np.sum(grid == TREE),
        "burning": np.sum(grid == FIRE),
        "burned": np.sum(grid == BURNED),
        "percent_burned": 100 * np.sum(grid == BURNED) / max(initial_trees, 1)
    })

    snapshots[tick] = grid.copy()

    return grid, pd.DataFrame(history), snapshots, initial_trees

final_grid, history, snapshots, initial_trees = run_simulation(
    width=width,
    height=height,
    density=density,
    probability_of_spread=probability_of_spread,
    south_wind_speed=south_wind_speed,
    west_wind_speed=west_wind_speed,
    big_jumps=big_jumps,
    max_ticks=max_ticks,
    seed=RANDOM_SEED
)

history.tail()

## 8. Resultado final

A figura abaixo mostra o estado final da simulação.

In [ ]:
plot_grid(final_grid, "Estado final da simulação")

## 9. Evolução temporal

Agora observamos a quantidade de árvores verdes, árvores em chamas e árvores queimadas ao longo do tempo.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["tick"], history["trees"], label="Árvores verdes")
plt.plot(history["tick"], history["burning"], label="Em chamas")
plt.plot(history["tick"], history["burned"], label="Queimadas")
plt.xlabel("Tick")
plt.ylabel("Quantidade de células")
plt.title("Evolução temporal da simulação")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["tick"], history["percent_burned"])
plt.xlabel("Tick")
plt.ylabel("Percentual queimado (%)")
plt.title("Percentual acumulado de árvores queimadas")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Snapshots da propagação

A sequência abaixo mostra alguns momentos da propagação do incêndio.

In [ ]:
for tick, snapshot in snapshots.items():
    plot_grid(snapshot, f"Snapshot — tick {tick}")

## 11. Métricas finais

As métricas finais ajudam a interpretar o comportamento global da simulação.

In [ ]:
final_metrics = {
    "árvores_iniciais": initial_trees,
    "árvores_restantes": int(np.sum(final_grid == TREE)),
    "árvores_queimadas": int(np.sum(final_grid == BURNED)),
    "percentual_queimado": float(100 * np.sum(final_grid == BURNED) / max(initial_trees, 1)),
    "ticks_executados": int(history["tick"].max()),
    "densidade": density,
    "probabilidade_base": probability_of_spread,
    "vento_sul": south_wind_speed,
    "vento_oeste": west_wind_speed,
    "big_jumps": big_jumps
}

pd.DataFrame([final_metrics])

## 12. Experimentos comparativos

Podemos comparar diferentes densidades da floresta para observar quando o fogo tende a atravessar grande parte do ambiente.

Essa análise se aproxima da ideia de **limiar crítico** ou **percolação**, comum em modelos de sistemas complexos.

In [ ]:
densities = [35, 45, 55, 65, 75]
results = []

for d in densities:
    final_grid_d, history_d, snapshots_d, initial_trees_d = run_simulation(
        width=width,
        height=height,
        density=d,
        probability_of_spread=probability_of_spread,
        south_wind_speed=south_wind_speed,
        west_wind_speed=west_wind_speed,
        big_jumps=big_jumps,
        max_ticks=max_ticks,
        seed=RANDOM_SEED
    )

    results.append({
        "density": d,
        "initial_trees": initial_trees_d,
        "burned": int(np.sum(final_grid_d == BURNED)),
        "remaining_trees": int(np.sum(final_grid_d == TREE)),
        "percent_burned": 100 * np.sum(final_grid_d == BURNED) / max(initial_trees_d, 1),
        "ticks": int(history_d["tick"].max())
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(results_df["density"], results_df["percent_burned"], marker="o")
plt.xlabel("Densidade inicial de árvores (%)")
plt.ylabel("Percentual queimado (%)")
plt.title("Efeito da densidade na propagação do fogo")
plt.grid(True, alpha=0.3)
plt.show()

## 13. Interpretação

A simulação evidencia que a propagação do fogo não depende apenas da presença de árvores, mas da conectividade espacial entre elas.

Em densidades baixas, o fogo encontra barreiras naturais, pois há muitas células vazias. Em densidades mais altas, a floresta se torna mais conectada, permitindo que o fogo atravesse grandes regiões.

O vento altera a assimetria da propagação, favorecendo algumas direções e dificultando outras. Quando `big_jumps` está ativado, a propagação pode ultrapassar pequenas barreiras, pois faíscas conseguem incendiar árvores mais distantes.

Esse tipo de modelo é útil para discutir:

- sistemas complexos;
- autômatos celulares;
- dinâmica espacial;
- limiares críticos;
- propagação em redes;
- simulações baseadas em agentes.

## 14. Próximas extensões possíveis

Algumas melhorias possíveis para pesquisa ou ensino:

1. incluir umidade do solo ou da vegetação;
2. variar o vento ao longo do tempo;
3. diferenciar tipos de vegetação;
4. medir se o fogo atravessou de um lado ao outro da grade;
5. repetir a simulação várias vezes por cenário;
6. calcular média, desvio padrão e intervalo de confiança;
7. exportar os resultados em CSV;
8. criar animação da propagação.